# Final Assignment: Analyzing Historical Stock/Revenue Data and Building a Dashboard

**Tesla (TSLA)** y **GameStop (GME)**: precio histórico (yfinance) + ingresos trimestrales (web scraping) + gráficos.

```
Descarga precios ──► Scraping de ingresos ──► Limpieza ──► Gráficos ──► Conclusiones
```

> Ejecútalo en un entorno con internet (IBM Skills Network Labs, Google Colab o tu Jupyter local).
> Cada celda de código indica a qué pregunta de la tarea corresponde.

In [ ]:
# Solo si faltan librerías (en el laboratorio de IBM suelen estar instaladas)
# %pip install yfinance beautifulsoup4 lxml html5lib pandas plotly requests

In [ ]:
import sys
sys.path.append("..")   # permite importar el paquete stock_revenue desde notebooks/

import pandas as pd
from stock_revenue.stocks import get_tesla_data, get_gme_data
from stock_revenue.scraping import (
    get_tesla_revenue, get_gme_revenue, TESLA_REVENUE_URL, GME_REVENUE_URL,
)
from stock_revenue.plotting import make_graph

pd.set_option("display.max_columns", 20)

## Pregunta 1 del notebook · Datos de Tesla con yfinance (1.2)
`yf.Ticker("TSLA").history(period="max")` devuelve la fecha como **índice**.
Con `reset_index()` pasa a ser una columna normal, necesaria para graficar y unir datos.

In [ ]:
tesla_data = get_tesla_data()
tesla_data.head()          # primeras cinco filas

## Pregunta 2 del notebook · Ingresos de Tesla con web scraping (1.3)
Flujo: `requests.get` → HTML → `BeautifulSoup` → tabla *Tesla Quarterly Revenue* → DataFrame → limpieza.

In [ ]:
tesla_revenue = get_tesla_revenue()
tesla_revenue.tail()       # últimas cinco filas

## Pregunta 3 del notebook · Datos de GameStop con yfinance (1.4)

In [ ]:
gme_data = get_gme_data()
gme_data.head()

## Pregunta 4 del notebook · Ingresos de GameStop con web scraping (1.5)

In [ ]:
gme_revenue = get_gme_revenue()
gme_revenue.tail()

## Pregunta 5 del notebook · Gráfico de Tesla (1.6)
Dos paneles con la misma escala temporal: precio de cierre (arriba) e ingresos (abajo).
Se filtra hasta junio de 2021, como pide el enunciado.

In [ ]:
make_graph(tesla_data, tesla_revenue, "Tesla").show()

## Pregunta 6 del notebook · Gráfico de GameStop (1.7)

In [ ]:
make_graph(gme_data, gme_revenue, "GameStop").show()

---
# Respuestas teóricas

### Cuestionario · Pregunta 2 · ¿Qué analizadores (parsers) permite Beautiful Soup para HTML?
`html.parser` (incluido en Python), `lxml` y `html5lib`. Se elige con el segundo argumento:
`BeautifulSoup(html, "html.parser")`. Para otro tipo de documento también existe `"xml"` (requiere lxml).
*Si en el examen aparece como opción "todos los anteriores", esa es la correcta.*

### Cuestionario · Pregunta 3 · ¿Por qué convertir la columna `Date` a `datetime`?
Porque como texto Python no sabe que son fechas: no puede ordenarlas cronológicamente,
restarlas, filtrarlas por rango (`<= '2021-06-14'`) ni dibujar un eje de tiempo correcto.
Con `pd.to_datetime` se habilitan todas esas operaciones.

### Cuestionario · Pregunta 4 · Quitar el símbolo `$` y las comas de `Price`/`Revenue`
```python
df["Revenue"] = df["Revenue"].str.replace(",|\$", "", regex=True)
```
`,|\$` es una expresión regular: "coma **o** signo dólar" (el `$` se escapa porque en regex significa fin de línea).

### Cuestionario · Pregunta 5 · Valor de `Revenue` en la **última** fila de `tesla_revenue`
Se obtiene con `tesla_revenue.tail(1)` o en la celda de la Pregunta 2 (la fila inferior).
El valor depende de la versión de la página que sirva el curso, así que **cópialo de tu salida**.

### Cuestionario · Pregunta 6 · Precio de cierre (`Close`) en la **primera** fila de `gme_data` (2002-02-13)
Se obtiene con `gme_data.head(1)["Close"]`. Cópialo de tu salida: yfinance puede devolver
precios ajustados por splits y varía ligeramente con la versión.

### Cuestionario · Pregunta 7 · Tendencia de Tesla entre 2018 y 2021
Tanto el **precio de la acción como los ingresos trimestrales crecieron de forma sostenida**
(con una subida de precio especialmente fuerte en 2020-2021). Comprueba en tu gráfico que
las dos curvas suben.

### Cuestionario · Pregunta 8 · Observación clave sobre GameStop
El **precio se disparó a principios de 2021** (pico muy puntual, ligado al episodio de ventas en corto
de enero de 2021) **sin un aumento equivalente de los ingresos**, que llevaban tiempo estables o a la baja.
Es decir: el precio dejó de reflejar la evolución del negocio.